# E-commerce Sales - Data Cleaning

**Objective:** Clean and prepare the raw data for analysis.

**Input:** Raw transaction data  
**Output:** Cleaned dataset ready for analysis

# 1. Setup

In [1]:
import duckdb
import pandas as pd

df1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
raw = pd.concat([df1, df2], ignore_index=True)


print(f"Loaded {len(raw):,} rows")

Loaded 1,067,371 rows


# 2. Standardise Column Names

In [2]:
raw = raw.rename(columns={'Customer ID': 'CustomerID'})

con = duckdb.connect()
con.register('raw_retail', raw)

print("Columns:", list(raw.columns))
print(f"Rows: {len(raw):,}")

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']
Rows: 1,067,371


## • Building CTE chain to clean raw data in stages, with each stage fixing one issue found in audit. 
## • Some observations will have more than 1 of the data quality issues identified. 
## • Before each stage, the number of data quality issues remaining will be counted to check if they have been removed in the previous stage

# 3. Remove Exact Duplicates

In [3]:
# Audit found 34,335 exact duplicate rows where all columns including the InvoiceDate timestamp are identical
# Will remove them as they represent a small percent of the total and cannot tell if genuine or extraction error.

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    )
    SELECT COUNT(*) AS no_duplication_total FROM deduped;
""").fetchdf()


,no_duplication_total
0,1033036


# 4. Flag Cancellations

#### Audit found 19,494 cancellations with C-prefix in invoice and 22,950 with negative quantities.

In [4]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * FROM raw_retail
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM deduped;
""").fetchdf()

,total_rows,cancellations,invalid_price,null_customer_id,null_description
0,1033036,22497.0,6019.0,235151.0,4275.0


In [5]:
# Flag Cancellations
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    )
    SELECT COUNT(*) AS rows_after_tag FROM cancellation_tag;
""").fetchdf()

,rows_after_tag
0,1033036


In [6]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS flagged_cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM cancellation_tag;
""").fetchdf()

,total_rows,flagged_cancellations,invalid_price,null_customer_id,null_description
0,1033036,22497.0,6019.0,235151.0,4275.0


# 5. Remove zero/negative prices

#### Audit found 6,207 with zero/negative prices

In [7]:
# remove zero/negative prices

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT COUNT(*) AS rows_after_filter FROM base_transactions;
""").fetchdf()

,rows_after_filter
0,1027017


In [8]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS flagged_cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM base_transactions;
""").fetchdf()

,total_rows,flagged_cancellations,invalid_price,null_customer_id,null_description
0,1027017,19104.0,0.0,229202.0,0.0


### Removing negative/zero prices also removed rows with null descriptions

# 6. Stock Code Cleaning

#### • Stock Code usually 5 or 6 characters long, composed of numbers and letters
#### • Will remove stock codes with less than 5 characters and also look closer at stock codes with no numbers

In [9]:
# Stock Code Distribution
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT
        LENGTH(CAST(StockCode AS VARCHAR)) AS code_length,
        COUNT(*) AS rows,
        COUNT(DISTINCT StockCode) AS distinct_codes,
        ANY_VALUE(StockCode) AS example
    FROM 
        base_transactions
    GROUP BY 
        code_length
    ORDER BY 
        rows DESC
""").fetchdf()

,code_length,rows,distinct_codes,example
0,5,897765,3401,21713
1,6,122113,1488,85049H
2,4,2113,3,POST
3,1,1660,5,M
4,3,1418,1,DOT
5,7,1336,8,15056BL
6,2,274,1,C2
7,12,178,8,BANK CHARGES
8,8,99,14,DCGS0003
9,9,61,3,AMAZONFEE


In [10]:
# Stock Codes without numbers
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT
        CAST(StockCode AS VARCHAR) AS code,
        ANY_VALUE(Description) AS description,
        COUNT(*) AS rows,
        ROUND(MIN(Price), 2) AS min_price,
        ROUND(MAX(Price), 2) AS max_price
    FROM 
        base_transactions
    WHERE 
        NOT REGEXP_MATCHES(CAST(StockCode AS VARCHAR), '[0-9]')
    GROUP BY 
        code
    """).fetchdf()

,code,description,rows,min_price,max_price
0,AMAZONFEE,AMAZON FEE,36,1.00,17836.46
1,DOT,DOTCOM POSTAGE,1418,0.35,4505.17
2,ADJUST,Adjustment by john on 26/01/2010 16,67,4.57,5117.03
3,DCGSSBOY,BOYS PARTY BAG,21,1.25,3.36
4,D,Discount,173,0.01,1867.86
5,DCGSSGIRL,GIRLS PARTY BAG,23,1.25,3.36
6,B,Adjust bad debt,1,11062.06,11062.06
7,CRUK,CRUK Commission,16,1.60,1100.44
8,PADS,PADS TO MATCH ALL CUSHIONS,18,0.00,36.60
9,S,SAMPLES,101,2.80,605.18


#### Stock codes without numbers don't represent products, will remove them and keep stock codes with length 5 or greater

In [11]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    )
    SELECT
        LENGTH(CAST(StockCode AS VARCHAR)) AS code_length,
        COUNT(*) AS rows,
        COUNT(DISTINCT StockCode) AS distinct_codes,
        ANY_VALUE(StockCode) AS example
    FROM 
        stock_code_without_numbers_length_adjust
    GROUP BY 
        code_length
    ORDER BY 
        rows DESC
    """).fetchdf()

,code_length,rows,distinct_codes,example
0,5,897765,3401,22195
1,6,122046,1487,79323P
2,7,1336,8,15056BL
3,8,99,14,DCGS0058
4,12,78,7,gift_0001_40
5,9,25,2,DCGSSGIRL


In [12]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    )
    SELECT
        *
    FROM 
        stock_code_without_numbers_length_adjust
    WHERE
        StockCode LIKE '%gift%'
    """).fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_cancellation
0,558066,gift_0001_50,Dotcomgiftshop Gift Voucher £50.00,1,2011-06-24 15:45:00,41.67,NaN,United Kingdom,0
1,491969,gift_0001_80,Dotcomgiftshop Gift Voucher £80.00,1,2009-12-14 17:57:00,69.56,NaN,United Kingdom,0
2,495343,gift_0001_30,Dotcomgiftshop Gift Voucher £30.00,1,2010-01-22 18:05:00,25.53,NaN,United Kingdom,0
3,496247,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00,1,2010-01-29 16:11:00,17.02,NaN,United Kingdom,0
4,562933,gift_0001_30,Dotcomgiftshop Gift Voucher £30.00,1,2011-08-10 16:51:00,25.00,NaN,United Kingdom,0
...,...,...,...,...,...,...,...,...,...
73,557500,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00,1,2011-06-20 15:27:00,16.67,NaN,United Kingdom,0
74,558614,gift_0001_10,Dotcomgiftshop Gift Voucher £10.00,1,2011-06-30 15:56:00,8.33,NaN,United Kingdom,0
75,508451,gift_0001_30,Dotcomgiftshop Gift Voucher £30.00,1,2010-05-14 16:33:00,25.53,NaN,United Kingdom,0
76,540238,gift_0001_30,Dotcomgiftshop Gift Voucher £30.00,1,2011-01-05 14:44:00,25.53,NaN,United Kingdom,0


#### Stock codes with 'gift' in the text are not products but vouchers so will remove them too

In [13]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    )
    SELECT
        *
    FROM
        stock_code_without_gift
    WHERE
        Description LIKE '%dot%'
    """).fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_cancellation


#### Audit found some products have more than 1 stock code representing it due to lower case and upper case variants

In [14]:
# Count of stock codes remaining that have lower case letters
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    )
    SELECT 
        COUNT(DISTINCT StockCode) AS raw_distinct
    FROM
        stock_code_without_gift
    WHERE 
        StockCode != UPPER(StockCode)
""").fetchdf()

,raw_distinct
0,173


In [15]:
# Count of stock codes remaining that have lower case variants

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    )
    SELECT
        UPPER(CAST(StockCode AS VARCHAR)) AS stock_code_upper,
        COUNT(DISTINCT StockCode) AS variants,
        COUNT(*) AS total_rows
    FROM 
        stock_code_without_gift
    GROUP BY 
        stock_code_upper
    HAVING 
        COUNT(DISTINCT StockCode) > 1
    ORDER BY 
        total_rows DESC
""").fetchdf()

,stock_code_upper,variants,total_rows
0,85123A,2,5744
1,85099B,2,4151
2,82494L,2,2125
3,85099C,2,1927
4,85099F,2,1901
...,...,...,...
165,90183C,2,14
166,37482B,2,10
167,84705B,2,5
168,84867A,2,5


#### • 170 out of the 173 reamining stock codes with lower case letters have a upper case variant. 
#### • Will convert all letters in stock codes to uppercase to avoid the same product being represented by two stock codes 

In [16]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    )
    SELECT
        UPPER(CAST(StockCode AS VARCHAR)) AS stock_code_upper,
        COUNT(DISTINCT StockCode) AS variants,
        COUNT(*) AS total_rows
    FROM 
        stock_code_variant_fix
    GROUP BY 
        stock_code_upper
    HAVING 
        COUNT(DISTINCT StockCode) > 1
    ORDER BY 
        total_rows DESC
""").fetchdf()

,stock_code_upper,variants,total_rows


#### Audit found 1,232 stock codes with more than one description

In [17]:
# How many stock codes still have multiple descriptions after cleaning so far

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    )
    SELECT 
        COUNT(*) AS affected_stock_codes
    FROM (
        SELECT StockCode
        FROM stock_code_variant_fix
        GROUP BY StockCode
        HAVING COUNT(DISTINCT Description) > 1
    )
""").fetchdf()

,affected_stock_codes
0,624


In [18]:
# Which stock codes are affected

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    ),
    normalised AS (
        SELECT
            StockCode,
            Description,
            UPPER(TRIM(Description)) AS description_clean
        FROM stock_code_variant_fix
    )
    SELECT
        StockCode,
        COUNT(DISTINCT Description) AS raw_variants,
        COUNT(DISTINCT description_clean) AS clean_variants
    FROM normalised
    GROUP BY StockCode
    HAVING COUNT(DISTINCT Description) > 1
    ORDER BY StockCode
""").fetchdf()

,StockCode,raw_variants,clean_variants
0,15058A,2,2
1,15058B,2,2
2,16011,2,1
3,16012,2,2
4,16151A,2,2
...,...,...,...
619,85232B,2,2
620,90014A,2,2
621,90014B,2,2
622,90014C,2,2


In [19]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    )
    SELECT
        StockCode,
        Description,
        Count(*)
    FROM
        stock_code_variant_fix
    WHERE
        StockCode = '15058A'
    GROUP BY
        StockCode,
        Description
""").fetchdf()

,StockCode,Description,count_star()
0,15058A,BLUE POLKADOT GARDEN PARASOL,93
1,15058A,BLUE WHITE SPOTS GARDEN PARASOL,62


In [20]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    )
    SELECT
        StockCode,
        Description,
        Count(*)
    FROM
        stock_code_variant_fix
    WHERE
        StockCode = '15058B'
    GROUP BY
        StockCode,
        Description
""").fetchdf()

,StockCode,Description,count_star()
0,15058B,PINK POLKADOT GARDEN PARASOL,102
1,15058B,PINK WHITE SPOTS GARDEN PARASOL,59


In [21]:
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    )
    SELECT
        StockCode,
        Description,
        Count(*)
    FROM
        stock_code_variant_fix
    WHERE
        StockCode = '16011'
    GROUP BY
        StockCode,
        Description
""").fetchdf()

,StockCode,Description,count_star()
0,16011,ANIMAL STICKERS,101
1,16011,ANIMAL STICKERS,12


In [22]:
# Out of the total how many are formatting noise (whitespace) vs relabelling

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    ),
    normalised AS (
        SELECT
            StockCode,
            Description,
            UPPER(TRIM(Description)) AS description_clean
        FROM stock_code_variant_fix
    ),
    summary AS (
        SELECT
            StockCode,
            COUNT(DISTINCT Description) AS raw_variants,
            COUNT(DISTINCT description_clean) AS clean_variants
        FROM normalised
        GROUP BY StockCode
        HAVING COUNT(DISTINCT Description) > 1
    )
    SELECT
        SUM(CASE WHEN clean_variants = 1 THEN 1 ELSE 0 END) AS formatting_only,
        SUM(CASE WHEN clean_variants > 1 THEN 1 ELSE 0 END) AS genuine_relabel,
        COUNT(*) AS total_affected
    FROM summary
""").fetchdf()

,formatting_only,genuine_relabel,total_affected
0,25.0,599.0,624


#### INSIGHT:
-  624 stock codes still carry multiple descriptions after existing cleaning
  
-  Of these, only 25 are formatting differences (whitespace). The other 599 are genuine relabelling e.g. "BLUE POLKADOT GARDEN PARASOL" vs "BLUE WHITE SPOTS GARDEN PARASOL" for the same product

-  The problem is mostly text based rather than formatting based and TRIM/UPPER cannot resolve it all therefore Description alone is not a reliable grouping key. Any product analysis should group by the StockCode and resolve to the most frequent description for that code

In [23]:
description_trimmed = con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    ),
    stock_code_without_numbers_length_adjust AS (
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    ),
    stock_code_without_gift AS (
        SELECT
            *
        FROM 
            stock_code_without_numbers_length_adjust
        WHERE
            StockCode NOT LIKE '%gift%'
    ),
    stock_code_variant_fix AS (
        SELECT
            Invoice,
            UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
            Description,
            Quantity,
            InvoiceDate,
            Price,
            CustomerID,
            Country
        FROM
            stock_code_without_gift
    )
    SELECT
        Invoice,
        StockCode,
        TRIM(Description) AS Description,
        Quantity,
        InvoiceDate,
        Price,
        CustomerID,
        Country
    FROM
        stock_code_variant_fix
""").fetchdf()

In [24]:
con.execute("""
    WITH normalised AS (
        SELECT
            StockCode,
            Description,
            UPPER(TRIM(Description)) AS description_clean
        FROM description_trimmed
    ),
    summary AS (
        SELECT
            StockCode,
            COUNT(DISTINCT Description) AS raw_variants,
            COUNT(DISTINCT description_clean) AS clean_variants
        FROM normalised
        GROUP BY StockCode
        HAVING COUNT(DISTINCT Description) > 1
    )
    SELECT
        SUM(CASE WHEN clean_variants = 1 THEN 1 ELSE 0 END) AS formatting_only,
        SUM(CASE WHEN clean_variants > 1 THEN 1 ELSE 0 END) AS genuine_relabel,
        COUNT(*) AS total_affected
    FROM summary
""").fetchdf()

,formatting_only,genuine_relabel,total_affected
0,1.0,599.0,600


In [25]:
# One of the Descriptions still has a formatting issue

con.execute("""
    WITH normalised AS (
        SELECT
            StockCode,
            Description,
            UPPER(TRIM(Description)) AS description_clean
        FROM description_trimmed
    )
    SELECT
        StockCode,
        COUNT(DISTINCT Description) AS raw_variants,
        COUNT(DISTINCT description_clean) AS clean_variants
    FROM normalised
    GROUP BY StockCode
    HAVING COUNT(DISTINCT Description) > 1 AND clean_variants = 1
    ORDER BY StockCode
""").fetchdf()

,StockCode,raw_variants,clean_variants
0,22619,2,1


In [26]:
# The stock code with remaining format issue
con.execute("""
    SELECT
        StockCode,
        Description,
        Count(*)
    FROM
        description_trimmed
    WHERE
        StockCode = '22619'
    GROUP BY
        StockCode,
        Description
""").fetchdf()

,StockCode,Description,count_star()
0,22619,SET OF 6 SOLDIER SKITTLES,716
1,22619,SET Of 6 SOLDIER SKITTLES,3


In [27]:
# This is the same description but one has a lower cased letters. 
# Will convert all descriptions to be upper case if there are no other descriptions in lower case
# Count of descriptions containing a lower case letter

con.execute("""
    SELECT COUNT(*) AS rows_with_lowercase
    FROM description_trimmed
    WHERE Description != UPPER(CAST(Description AS VARCHAR))
""").fetchdf()

,rows_with_lowercase
0,3136


In [28]:
# Example of descriptions with a lower case
con.execute("""
    SELECT *
    FROM description_trimmed
    WHERE Description != UPPER(CAST(Description AS VARCHAR))
""").fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,563351,22686,FRENCH BLUE METAL DOOR SIGN No,20,2011-08-15 14:17:00,1.25,16902.0,United Kingdom
1,C564217,22965,3 TRADITIONAl BISCUIT CUTTERS SET,-1,2011-08-24 09:24:00,2.10,12994.0,United Kingdom
2,564661,21703,BAG 125g SWIRLY MARBLES,12,2011-08-26 14:45:00,0.42,12948.0,United Kingdom
3,565141,46000U,POLYESTER FILLER PAD 30CMx30CM,1,2011-09-01 13:40:00,1.25,17841.0,United Kingdom
4,565231,22686,FRENCH BLUE METAL DOOR SIGN No,1,2011-09-02 09:26:00,2.46,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
3131,498725,46000P,POLYESTER FILLER PAD 65CMx65CM,4,2010-02-22 15:12:00,4.25,14667.0,United Kingdom
3132,555737,21703,BAG 125g SWIRLY MARBLES,24,2011-06-07 09:19:00,0.42,16153.0,United Kingdom
3133,560538,22965,3 TRADITIONAl BISCUIT CUTTERS SET,12,2011-07-19 12:19:00,2.10,16128.0,United Kingdom
3134,561078,22965,3 TRADITIONAl BISCUIT CUTTERS SET,12,2011-07-24 16:05:00,2.10,16178.0,United Kingdom


#### Converting all descriptions to upper case will fix issue with the stockcode 22619's description 
#### Other descriptions with lower case are mostly due to labels for measurements like grams, centrimetres. T
#### These labels being in upper or lower case won't change the interpretation

In [29]:
description_upper = con.execute("""
    SELECT 
        Invoice,
        StockCode,
        UPPER(Description) AS Description,
        Quantity,
        InvoiceDate,
        Price,
        CustomerID,
        Country
    FROM 
        description_trimmed
""").fetchdf()

In [30]:
con.execute("""
    WITH normalised AS (
        SELECT
            StockCode,
            Description,
            UPPER(TRIM(Description)) AS description_clean
        FROM description_upper
    ),
    summary AS (
        SELECT
            StockCode,
            COUNT(DISTINCT Description) AS raw_variants,
            COUNT(DISTINCT description_clean) AS clean_variants
        FROM normalised
        GROUP BY StockCode
        HAVING COUNT(DISTINCT Description) > 1
    )
    SELECT
        SUM(CASE WHEN clean_variants = 1 THEN 1 ELSE 0 END) AS formatting_only,
        SUM(CASE WHEN clean_variants > 1 THEN 1 ELSE 0 END) AS genuine_relabel,
        COUNT(*) AS total_affected
    FROM summary
""").fetchdf()

,formatting_only,genuine_relabel,total_affected
0,0.0,599.0,599


# 7. Tag Guests

#### • 228,488 observations (22.7% of total) remaining have no CustomerID, their revenue is valid but will be filtered out for the customer analysis
#### • Null values in Description were found in audit but have been removed in stage 5

In [45]:
# Tag rows with no CustomerID as guest transactions
con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM description_upper
    )
    SELECT
        SUM(is_guest) AS guest_rows,
        SUM(1 - is_guest) AS registered_rows,
        COUNT(*) AS total_rows
    FROM guest_tag;
    """).fetchdf()

,guest_rows,registered_rows,total_rows
0,227167.0,794182.0,1021349


In [31]:
# Remaining data quality issues

con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM description_upper
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS flagged_cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(is_guest) AS guest_total,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM guest_tag;
""").fetchdf()

,total_rows,flagged_cancellations,invalid_price,null_customer_id,guest_total,null_description
0,1021271,17918.0,0.0,227089.0,227089.0,0.0


# 8. Derive Revenue

In [32]:
con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM description_upper
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    )
    SELECT
        MIN(Revenue) AS min_revenue,
        MAX(Revenue) AS max_revenue,
        ROUND(AVG(Revenue), 2) AS avg_revenue,
        ROUND(SUM(Revenue), 2) AS total_revenue,
        MIN(Quantity) AS min_quantity,
        MAX(Quantity) AS max_quantity,
        ROUND(AVG(Quantity), 2) AS avg_quantity,
        ROUND(SUM(Quantity), 2) AS total_quantity,
        MIN(Price) AS min_price,
        MAX(Price) AS max_price,
        ROUND(AVG(Price), 2) AS avg_price,
        ROUND(SUM(Price), 2) AS total_price,
        COUNT(*) total_rows
    FROM revenue_derive;
    """).fetchdf()

,min_revenue,max_revenue,avg_revenue,total_revenue,min_quantity,max_quantity,avg_quantity,total_quantity,min_price,max_price,avg_price,total_price,total_rows
0,-168469.6,168469.6,18.53,18928370.2,-80995,80995,10.5,10720354.0,0.03,1157.15,3.36,3434273.8,1021271


# 9. Date Features

In [33]:
con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM description_upper
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    ),
    cleaned AS (
        SELECT *,
            YEAR(InvoiceDate) AS Year,
            MONTH(InvoiceDate) AS Month,
            DAYOFWEEK(InvoiceDate) AS DayOfWeek,
            HOUR(InvoiceDate) AS Hour,
            CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
            DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
            DATE_TRUNC('month', InvoiceDate) AS MonthStart
        FROM revenue_derive
    )
    SELECT * FROM cleaned LIMIT 5;
    """).fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_guest,Revenue,Year,Month,DayOfWeek,Hour,IsWeekend,WeekStart,MonthStart
0,562619,23167,SMALL CERAMIC TOP STORAGE JAR,11,2011-08-08 12:34:00,0.83,15311.0,United Kingdom,0,9.13,2011,8,1,12,0,2011-08-08,2011-08-01
1,562619,23196,VINTAGE LEAF MAGNETIC NOTEPAD,1,2011-08-08 12:34:00,1.45,15311.0,United Kingdom,0,1.45,2011,8,1,12,0,2011-08-08,2011-08-01
2,562621,20725,LUNCH BAG RED RETROSPOT,10,2011-08-08 12:36:00,1.65,14415.0,United Kingdom,0,16.50,2011,8,1,12,0,2011-08-08,2011-08-01
3,562621,21928,JUMBO BAG SCANDINAVIAN BLUE PAISLEY,10,2011-08-08 12:36:00,2.08,14415.0,United Kingdom,0,20.80,2011,8,1,12,0,2011-08-08,2011-08-01
4,562649,21947,SET OF 6 HEART CHOPSTICKS,1,2011-08-08 13:11:00,1.25,14606.0,United Kingdom,0,1.25,2011,8,1,13,0,2011-08-08,2011-08-01


# 8. Validate Final Dataset

In [34]:
# Counting number of observations per data cleaning step
con.execute("""
    WITH deduped AS (
            SELECT DISTINCT * 
            FROM raw_retail
        ),
        cancellation_tag AS (
            SELECT *,
                CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                     OR Quantity < 0 
                     THEN 1 ELSE 0 END AS is_cancellation
            FROM deduped
        ),
        base_transactions AS (
            SELECT * 
            FROM cancellation_tag
            WHERE Price > 0
        ),
        stock_code_without_numbers_length_adjust AS (
            SELECT *
            FROM 
                (SELECT *
                    FROM base_transactions
                    WHERE LENGTH(StockCode) >= 5
                )
            WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
        ),
        stock_code_without_gift AS (
            SELECT *
            FROM stock_code_without_numbers_length_adjust
            WHERE StockCode NOT LIKE '%gift%'
        ),
        stock_code_variant_fix AS (
            SELECT
                Invoice,
                UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
                Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM stock_code_without_gift
        ),
        description_trimmed AS (
            SELECT
                Invoice,
                StockCode,
                TRIM(Description) AS Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM stock_code_variant_fix
        ),
        description_upper AS (
            SELECT 
                Invoice,
                StockCode,
                UPPER(Description) AS Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM description_trimmed
        ),
        guest_tag AS (
            SELECT *,
                CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
            FROM description_upper
        ),
        revenue_derive AS (
            SELECT 
                *,
                Quantity * Price AS Revenue
            FROM guest_tag
        ),
        cleaned AS (
            SELECT *,
                YEAR(InvoiceDate) AS Year,
                MONTH(InvoiceDate) AS Month,
                DAYOFWEEK(InvoiceDate) AS DayOfWeek,
                HOUR(InvoiceDate) AS Hour,
                CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
                DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
                DATE_TRUNC('month', InvoiceDate) AS MonthStart
            FROM revenue_derive
        )
    SELECT 'raw' AS stage, COUNT(*) AS num_of_rows FROM raw_retail
    UNION ALL SELECT 'deduped', COUNT(*) FROM deduped
    UNION ALL SELECT 'flagged_cancellations', COUNT(*) FROM cancellation_tag
    UNION ALL SELECT 'base_transactions', COUNT(*) FROM base_transactions
    UNION ALL SELECT 'stock_code_without_numbers_length_adjust', COUNT(*) FROM stock_code_without_numbers_length_adjust
    UNION ALL SELECT 'stock_code_without_gift', COUNT(*) FROM stock_code_without_gift
    UNION ALL SELECT 'stock_code_variant_fix', COUNT(*) FROM stock_code_variant_fix
    UNION ALL SELECT 'description_trimmed', COUNT(*) FROM description_trimmed
    UNION ALL SELECT 'description_upper', COUNT(*) FROM description_upper
    UNION ALL SELECT 'with_guest_tag', COUNT(*) FROM guest_tag
    UNION ALL SELECT 'with_revenue', COUNT(*) FROM revenue_derive
    UNION ALL SELECT 'cleaned', COUNT(*) FROM cleaned;
    """).fetchdf()

,stage,num_of_rows
0,raw,1067371
1,deduped,1033036
2,flagged_cancellations,1033036
3,base_transactions,1027017
4,stock_code_without_numbers_length_adjust,1021349
5,stock_code_without_gift,1021271
6,stock_code_variant_fix,1021271
7,description_trimmed,1021271
8,description_upper,1021271
9,with_guest_tag,1021271


In [35]:
# Checking if any data quality issues still exist
con.execute("""
    WITH deduped AS (
            SELECT DISTINCT * 
            FROM raw_retail
        ),
        cancellation_tag AS (
            SELECT *,
                CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                     OR Quantity < 0 
                     THEN 1 ELSE 0 END AS is_cancellation
            FROM deduped
        ),
        base_transactions AS (
            SELECT * 
            FROM cancellation_tag
            WHERE Price > 0
        ),
        stock_code_without_numbers_length_adjust AS (
            SELECT *
            FROM 
                (SELECT *
                    FROM base_transactions
                    WHERE LENGTH(StockCode) >= 5
                )
            WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
        ),
        stock_code_without_gift AS (
            SELECT *
            FROM stock_code_without_numbers_length_adjust
            WHERE StockCode NOT LIKE '%gift%'
        ),
        stock_code_variant_fix AS (
            SELECT
                Invoice,
                UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
                Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM stock_code_without_gift
        ),
        description_trimmed AS (
            SELECT
                Invoice,
                StockCode,
                TRIM(Description) AS Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM stock_code_variant_fix
        ),
        description_upper AS (
            SELECT 
                Invoice,
                StockCode,
                UPPER(Description) AS Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM description_trimmed
        ),
        guest_tag AS (
            SELECT *,
                CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
            FROM description_upper
        ),
        revenue_derive AS (
            SELECT 
                *,
                Quantity * Price AS Revenue
            FROM guest_tag
        ),
        cleaned AS (
            SELECT *,
                YEAR(InvoiceDate) AS Year,
                MONTH(InvoiceDate) AS Month,
                DAYOFWEEK(InvoiceDate) AS DayOfWeek,
                HOUR(InvoiceDate) AS Hour,
                CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
                DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
                DATE_TRUNC('month', InvoiceDate) AS MonthStart
            FROM revenue_derive
        )
    SELECT
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS bad_price,
        SUM(CASE WHEN is_cancellation NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_cancellation_flag,
        SUM(CASE WHEN LENGTH(StockCode) < 5 
                   OR StockCode IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE') THEN 1 ELSE 0 END) AS bad_stock_code,
        SUM(CASE WHEN StockCode != UPPER(StockCode) THEN 1 ELSE 0 END) AS bad_stock_code_case,
        SUM(CASE WHEN Description != TRIM(Description) THEN 1 ELSE 0 END) AS bad_description_whitespace,
        SUM(CASE WHEN Description != UPPER(Description) THEN 1 ELSE 0 END) AS bad_description_case,
        SUM(CASE WHEN IsWeekend NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_weekend_flag,
        SUM(CASE WHEN is_guest NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_guest_flag
    FROM cleaned;
    """).fetchdf()

,bad_price,bad_cancellation_flag,bad_stock_code,bad_stock_code_case,bad_description_whitespace,bad_description_case,bad_weekend_flag,bad_guest_flag
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 9. Saving cleaned data

In [36]:
con.execute("""
    CREATE OR REPLACE VIEW cleaned AS
    WITH deduped AS (
            SELECT DISTINCT * 
            FROM raw_retail
        ),
        cancellation_tag AS (
            SELECT *,
                CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                     OR Quantity < 0 
                     THEN 1 ELSE 0 END AS is_cancellation
            FROM deduped
        ),
        base_transactions AS (
            SELECT * 
            FROM cancellation_tag
            WHERE Price > 0
        ),
        stock_code_without_numbers_length_adjust AS (
            SELECT *
            FROM 
                (SELECT *
                    FROM base_transactions
                    WHERE LENGTH(StockCode) >= 5
                )
            WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
        ),
        stock_code_without_gift AS (
            SELECT *
            FROM stock_code_without_numbers_length_adjust
            WHERE StockCode NOT LIKE '%gift%'
        ),
        stock_code_variant_fix AS (
            SELECT
                Invoice,
                UPPER(CAST(StockCode AS VARCHAR)) AS StockCode,
                Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM stock_code_without_gift
        ),
        description_trimmed AS (
            SELECT
                Invoice,
                StockCode,
                TRIM(Description) AS Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM stock_code_variant_fix
        ),
        description_upper AS (
            SELECT 
                Invoice,
                StockCode,
                UPPER(Description) AS Description,
                Quantity,
                InvoiceDate,
                Price,
                CustomerID,
                Country,
                is_cancellation
            FROM description_trimmed
        ),
        guest_tag AS (
            SELECT *,
                CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
            FROM description_upper
        ),
        revenue_derive AS (
            SELECT 
                *,
                Quantity * Price AS Revenue
            FROM guest_tag
        )
    SELECT *,
        YEAR(InvoiceDate) AS Year,
        MONTH(InvoiceDate) AS Month,
        DAYOFWEEK(InvoiceDate) AS DayOfWeek,
        HOUR(InvoiceDate) AS Hour,
        CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
        DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
        DATE_TRUNC('month', InvoiceDate) AS MonthStart
    FROM revenue_derive;
""")

In [37]:
# All valid sales data including guests
con.execute("""
    COPY (SELECT * FROM cleaned)
    TO '../data/processed/cleaned.csv' (HEADER, DELIMITER ',')
""")

In [38]:
# Registered customers only
con.execute("""
    COPY (SELECT * FROM cleaned WHERE is_guest = 0)
    TO '../data/processed/registered_only.csv' (HEADER, DELIMITER ',');
""")

# 10. Cleaning Summary

In [44]:
stats = con.execute("""
    SELECT
        COUNT(*) AS final_rows,
        COUNT(DISTINCT CustomerID) AS unique_customers,
        COUNT(DISTINCT StockCode) AS unique_products,
        COUNT(DISTINCT Country) AS unique_countries,
        ROUND(SUM(Revenue), 2) AS total_revenue
    FROM cleaned
""").fetchdf()

raw_rows    = len(raw)
final_rows  = stats['final_rows'][0]
removed     = raw_rows - final_rows
pct_removed = removed / raw_rows * 100
customers   = stats['unique_customers'][0]
products    = stats['unique_products'][0]
countries   = stats['unique_countries'][0]
revenue     = stats['total_revenue'][0]

print('=' * 60)
print('CLEANING SUMMARY')
print('=' * 60)
print(f"""
ROWS:
  Original:  {raw_rows:,}
  Final:     {final_rows:,}
  Removed:   {removed:,} ({pct_removed:.1f}%)

WHAT WAS REMOVED OR EDITED:
  • Exact row duplicates removed. With no line item ID we can't tell whether these are extraction errors or genuine repeated entries. 
      Audit found they are a small share of both rows and revenue and are removed to avoid risk of counting the same sale twice
  
  • Cancellations represented with 'C' prefix or negative quantities were flagged
  • Zero price removed as these were bad debt entries and negative price removed due to no clear meaning to them
  • Stock codes under 5 characters, admin codes (BANK CHARGES, ADJUST, AMAZONFEE) and gift voucher codes removed as they are not products
  • Stock codes uppercased so the same product is not split across stock code variants
  • Descriptions trimmed and uppercased so the same product is not split across formatting variants

FINAL DATASET (cleaned.csv):
  - Rows: {final_rows:,}
  - Customers: {customers:,} (registered only, guests excluded from count)
  - Products: {products:,}
  - Countries: {countries:,}
  - Total Revenue: GBP {revenue:,.2f}

OUTPUT FILES:
  • cleaned.csv ---> all valid sales including guests
  • registered_only.csv ---> valid sales with CustomerID for cohort and RFM

Ready for analysis.ipynb
""")

CLEANING SUMMARY

ROWS:
  Original:  1,067,371
  Final:     1,021,271
  Removed:   46,100 (4.3%)

WHAT WAS REMOVED OR EDITED:
  • Exact row duplicates removed. With no line item ID we can't tell whether these are extraction errors or genuine repeated entries. 
      Audit found they are a small share of both rows and revenue and are removed to avoid risk of counting the same sale twice

  • Cancellations represented with 'C' prefix or negative quantities were flagged
  • Zero price removed as these were bad debt entries and negative price removed due to no clear meaning to them
  • Stock codes under 5 characters, admin codes (BANK CHARGES, ADJUST, AMAZONFEE) and gift voucher codes removed as they are not products
  • Stock codes uppercased so the same product is not split across stock code variants
  • Descriptions trimmed and uppercased so the same product is not split across formatting variants

FINAL DATASET (cleaned.csv):
  - Rows: 1,021,271
  - Customers: 5,875 (registered only, g